# Experiment008 — Four ensembles from cached out-of-fold predictions

**Kaggle: Predicting Credit Card Fraud (S6E11)** · **Primary metric: out-of-fold (OOF) ROC AUC** · **No re-running the original 19-candidate sweep**.

This is a **second notebook** that reads the files produced by `credit_card_fraud.ipynb` in `final_project/outputs/`. It reuses existing predictions and saved logistic-regression fold models. It trains **only two new second-stage components**: (i) an XGBoost *regressor on explicit probability residuals* and (ii) a random-forest *stacking meta-classifier*. Existing logistic-offset boosting is loaded, **not retrained**. This notebook does **not** modify the original notebook or its results.

### Four candidates and the benchmark

| Strategy | What gets combined? | Requires training here? |
|---|---|---|
| **Benchmark — best standalone** | Best OOF ROC AUC among non-offset standalone candidates | No |
| **Ensemble 1 — probability blending** | Equal-weight mean of the best standalone model from each family | No |
| **Ensemble 2 — logistic-offset XGBoost** | Logistic logits + trained boosted-tree corrections | No (cached) |
| **Ensemble 3 — explicit residual XGBoost** | Logistic probability + prediction of `y − p_logit`, clipped to [0,1] | Yes: 5 XGBoost regressors |
| **Ensemble 4 — RF stacking** | Random forest trained *only* on cached base-model probability vectors | Yes: RF meta CV and final refit |

**Important terminology:** probability blending is not classical bootstrap bagging. Logistic-offset boosting is not stacking. RF stacking truly trains a second-level learner on base predictions. The residual approach is deliberately distinct from the logit-offset approach.

**Evaluation limitation:** all candidate definitions/weights selected by original OOF AUC and second-level meta-CV use the same development dataset; final selection is exploratory and optimistically biased. Moreover, recombining existing five-fold OOF predictions does **not** reproduce a fully nested outer-CV evaluation of stacking: some base-model training folds include records in a meta-validation fold. Use an untouched holdout or newly nested CV for unbiased performance claims. Do **not** report the apparent AUC winner as an independently verified gain.

## 1. Configuration and environment

Run from `final_project/` (preferred) using the **Python (LDS1071 Fraud)** Conda kernel. **Do not delete or overwrite the existing `outputs/` directory.** This notebook reads it and creates its own subdirectory: `outputs/ensemble_extension/`. XGBoost uses GPU when a real CUDA training probe succeeds; RF stacking uses CPU and `n_jobs=-1`.

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import time
import warnings
import zipfile
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost
from xgboost import XGBRegressor
from sklearn import __version__ as sklearn_version
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             log_loss, brier_score_loss, roc_curve, precision_recall_curve)
from sklearn.model_selection import StratifiedKFold, train_test_split
from IPython.display import display

RANDOM_STATE = 42
N_JOBS = -1
# Set to 'cpu' if GPU probing / runtime has problems. 'auto' probes actual CUDA training.
RESIDUAL_XGB_DEVICE = 'auto'   # 'auto' | 'cpu' | 'cuda'
RESIDUAL_XGB_GPU_ID = '0'
RESIDUAL_XGB_PARAMS = dict(n_estimators=220, max_depth=3,
                           learning_rate=0.045, min_child_weight=7,
                           subsample=0.85, colsample_bytree=0.85,
                           reg_lambda=5.0, objective='reg:squarederror')
RF_META_PARAMS = dict(n_estimators=280, max_depth=6,
                      min_samples_leaf=75, max_features='sqrt',
                      bootstrap=True, max_samples=0.85, n_jobs=-1)
STACKING_BASE_MODE = 'all_standalone'  # or 'best_per_family'
BASELINE_LOGISTIC = 'logit_C1'
PREFERRED_OFFSET = 'offset_xgb_depth3_lr05'
USE_SAVED_SECOND_STAGE_MODELS = True
RERUN_EXISTING_SECOND_STAGE = False     # Change to True to refit our *new* stages only.
REBUILD_MISSING_LOGISTIC = False          # Default: NEVER retrain the original base model.
BOOTSTRAP_REPS = 100                    # Paired illustrative AUC-difference intervals.
PLOT_DPI = 160

if (Path.cwd() / 'credit_card_fraud.ipynb').is_file():
    PROJECT_DIR = Path.cwd()
elif (Path.cwd() / 'final_project' / 'credit_card_fraud.ipynb').is_file():
    PROJECT_DIR = Path.cwd() / 'final_project'
else:
    raise FileNotFoundError('Open this notebook from the repository root or final_project directory.')
# The first notebook wrote relative to its *kernel working directory*.
# Search both likely locations to accommodate Jupyter/VS Code on Windows.
original_output_candidates = list(dict.fromkeys([
    PROJECT_DIR / 'outputs', PROJECT_DIR.parent / 'outputs', Path.cwd() / 'outputs'
]))
required_cache_names = ('all_candidate_predictions.npz', 'candidate_metrics.csv',
                        'run_manifest.json')
OUTPUT_DIR = next((candidate for candidate in original_output_candidates
                   if all((candidate / name).is_file() for name in required_cache_names)),
                  original_output_candidates[0])
# Always store *new* outputs under final_project, even if base models are elsewhere.
EXT_DIR = PROJECT_DIR / 'outputs' / 'ensemble_extension'
CACHE_DIR = EXT_DIR / 'models'
FIG_DIR = EXT_DIR / 'figures'
for directory in [EXT_DIR, CACHE_DIR, FIG_DIR]: directory.mkdir(parents=True, exist_ok=True)
sns.set_theme(style='whitegrid', context='notebook', palette='colorblind')
plt.rcParams.update({'savefig.dpi':PLOT_DPI, 'axes.titleweight':'semibold'})
print('Project:', PROJECT_DIR.resolve())
print('Existing trained outputs:', OUTPUT_DIR.resolve())
print('New experiment outputs:', EXT_DIR.resolve())
print('scikit-learn', sklearn_version, '| xgboost', xgboost.__version__)

## 2. Locate and validate cached results, without training

Expected original paths:

```text
outputs/all_candidate_predictions.npz
outputs/candidate_metrics.csv
outputs/run_manifest.json
outputs/models/logit_C1_fold1.joblib  ...  logit_C1_fold5.joblib
```

`all_candidate_predictions.npz` stores **one OOF probability and one mean test probability** for each candidate. Each OOF observation was predicted by a model fitted without that observation. The exact row order and original 5-fold partition must be preserved. We reconstruct folds using `StratifiedKFold(5, shuffle=True, random_state=42)`; we will cross-check each saved logistic fold model's predictions against the stored OOF values before using them.

Only the original experiment's raw files are missing from this deliverable ZIP: they should already exist on your workstation. Their absence here does **not** mean this notebook has run the full experiment.

In [ ]:
required = ['all_candidate_predictions.npz', 'candidate_metrics.csv', 'run_manifest.json']
missing = [name for name in required if not (OUTPUT_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(
        f'Missing original full-run output files in {OUTPUT_DIR}: {missing}. '
        'The slides ZIP contains figures, not the models/OOF cache. '
        'Use the SAME outputs/ folder created by your original full notebook run.'
    )
manifest = json.loads((OUTPUT_DIR / 'run_manifest.json').read_text(encoding='utf-8'))
assert manifest['n_splits'] == 5 and manifest['random_state'] == 42, (
    'The saved run used different folds. Stop and inspect rather than assuming alignment.')

DATA_DIRS = [PROJECT_DIR / 'data', PROJECT_DIR, Path('/kaggle/input/playground-series-s6e11')]
ZIP_CANDIDATES = [PROJECT_DIR / 'playground-series-s6e11.zip',
                  PROJECT_DIR / 'data' / 'playground-series-s6e11.zip',
                  PROJECT_DIR.parent / 'playground-series-s6e11.zip']

def read_competition_csv(filename):
    for folder in DATA_DIRS:
        if (folder / filename).is_file():
            return pd.read_csv(folder / filename)
    for compressed in ZIP_CANDIDATES:
        if compressed.is_file():
            with zipfile.ZipFile(compressed) as z:
                with z.open(filename) as stream:
                    return pd.read_csv(stream)
    raise FileNotFoundError(f'Cannot locate {filename}. Put the original Kaggle ZIP beside the notebook.')

train_full = read_competition_csv('train.csv')
test = read_competition_csv('test.csv')
sample_submission = read_competition_csv('sample_submission.csv')
TARGET, ID = 'is_fraud', 'id'
if manifest.get('fast_dev_run', False):
    # Reconstruct the *original* development stratified sample exactly.
    n_dev = int(manifest['training_rows'])
    train, _ = train_test_split(train_full, train_size=n_dev,
                                stratify=train_full[TARGET], random_state=RANDOM_STATE)
    train = train.reset_index(drop=True)
    warnings.warn('Development-cache mode: results will NOT be full-dataset evaluations.')
else:
    train = train_full
assert len(train) == int(manifest['training_rows']), 'Training size disagrees with saved run.'
assert len(test) == int(manifest['test_rows']), 'Test size disagrees with saved run.'
assert not train[ID].duplicated().any() and not test[ID].duplicated().any()
assert sample_submission[ID].equals(test[ID]), 'Kaggle test ID/order mismatch.'
assert train[TARGET].isin([0,1]).all()
feature_cols = [col for col in train.columns if col not in (ID, TARGET)]
X = train[feature_cols]
y = train[TARGET].astype(int).to_numpy()
X_test = test[feature_cols]
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
fold_splits = list(cv.split(X,y))
fold_id = np.zeros(len(y), dtype=np.uint8)
for f, (_, valid) in enumerate(fold_splits,1): fold_id[valid] = f
assert fold_id.min() == 1 and fold_id.max() == 5

scores = pd.read_csv(OUTPUT_DIR / 'candidate_metrics.csv')
assert {'candidate','family','roc_auc'}.issubset(scores.columns)
with np.load(OUTPUT_DIR / 'all_candidate_predictions.npz', allow_pickle=False) as cache:
    predictions = {}
    for name in scores['candidate']:
        oof_key, test_key = f'{name}__oof', f'{name}__test'
        if oof_key not in cache or test_key not in cache:
            raise KeyError(f'Missing cached prediction for {name}. Cache may be incomplete.')
        p_oof = np.asarray(cache[oof_key],dtype=np.float64)
        p_test = np.asarray(cache[test_key],dtype=np.float64)
        assert p_oof.shape == (len(y),) and p_test.shape == (len(test),), name
        assert np.isfinite(p_oof).all() and np.isfinite(p_test).all(), name
        assert min(p_oof.min(),p_test.min()) >= -1e-6, name
        assert max(p_oof.max(),p_test.max()) <= 1+1e-6, name
        predictions[name] = {'oof':np.clip(p_oof,0,1), 'test':np.clip(p_test,0,1)}
# Verify the original scoreboard agrees with saved individual OOF predictions.
for record in scores.itertuples():
    observed = roc_auc_score(y, predictions[record.candidate]['oof'])
    if abs(observed - float(record.roc_auc)) > 2e-5:
        raise AssertionError(f'AUC cache mismatch for {record.candidate}: {observed} vs {record.roc_auc}')
print('Cache verified:',len(y),'train rows,',len(test),'test rows,',len(predictions),'candidates')
print('Original run manifest:', manifest)
display(scores.sort_values('roc_auc',ascending=False).head(10))

## 3. Define the benchmark and first two ensembles (no fitting)

**Standalone** means one fitted model family with its own predictions (logistic, PCA/spline logistic, random forest, XGBoost, CatBoost, or their tuned variants). We exclude **logistic-offset XGBoost** from the standalone category, because it is already a two-stage ensemble.

**Ensemble 1 — blending** uses equal weights on the best standalone candidate from each family, selected using existing OOF scores. For a set of candidate predictions \(p_m(x)\),

\[p_{\mathrm{blend}}(x)=\frac1{|S|}\sum_{m\in S}p_m(x).\]

**Ensemble 2 — logistic-offset boosting** has already been trained: \(p(x)=\sigma(f_{\mathrm{LR}}(x)+g_{\mathrm{XGB}}(x))\). XGBoost learns a *logit correction* from the binary logistic loss, with the logistic logits supplied as `base_margin`. It does **not** fit `y - p` as a separate regression target.

In [ ]:
# Family names and full OOF scores come from the previous completed run.
standalone = scores.loc[~scores['family'].eq('logistic_offset_xgb')].copy()
offset_available = scores.loc[scores['family'].eq('logistic_offset_xgb')].copy()
assert len(standalone) >= 2, 'Need at least two fitted standalone candidates.'
if offset_available.empty:
    raise ValueError('The original run contains no logistic_offset_xgb candidate; cannot compare four ensembles.')
standalone = standalone.sort_values(['roc_auc','candidate'],ascending=[False,True]).reset_index(drop=True)
best_single = str(standalone.iloc[0]['candidate'])
family_members = (standalone.drop_duplicates('family',keep='first')['candidate'].astype(str).tolist())
assert len(family_members)>=2
if PREFERRED_OFFSET in offset_available['candidate'].values:
    offset_name = PREFERRED_OFFSET
else:
    offset_name = str(offset_available.sort_values('roc_auc',ascending=False).iloc[0]['candidate'])
if BASELINE_LOGISTIC not in predictions:
    available_lrs = standalone.loc[standalone['family'].eq('logistic'),'candidate'].tolist()
    if not available_lrs: raise ValueError('A cached logistic baseline is required for explicit residual modeling.')
    BASELINE_LOGISTIC = available_lrs[0]
    warnings.warn(f'Using {BASELINE_LOGISTIC} as residual-model baseline because logit_C1 was absent.')

results = {
    'best_single': predictions[best_single],
    'ensemble1_blend': {
        'oof':np.mean([predictions[n]['oof'] for n in family_members],axis=0),
        'test':np.mean([predictions[n]['test'] for n in family_members],axis=0)},
    'ensemble2_logit_offset': predictions[offset_name],
}
print('Best standalone:',best_single)
print('Blending members:',family_members)
print('Offset ensemble:',offset_name)
print('Residual logistic baseline:',BASELINE_LOGISTIC)

## 4. Ensemble 3 — explicit residual XGBoost (new experiment)

This is **exactly the explicit error-correction idea**, distinct from `base_margin`:

1. Load the previously saved **fold-specific logistic pipeline** \(L_f\).
2. Predict on the **training part of fold** \(f\), \(p_{\mathrm{LR},i}\). Compute observed probability residuals \(r_i=y_i-p_{\mathrm{LR},i}\).
3. Fit a new **XGBRegressor** with squared-error loss to \((X_i,r_i)\), using the **same cached fold preprocessing** from the logistic pipeline. For simplicity, use `preprocess.transform(X)` from the saved fold model rather than refitting a transformer.
4. On held-out fold \(f\), compute \(p_{\mathrm{corrected}}=\operatorname{clip}(p_{\mathrm{LR}}+\hat r,\varepsilon,1-\varepsilon)\).
5. Average the five independently fitted fold corrections for the Kaggle test predictions.

\[\boxed{\hat p_{\mathrm{residual}}(x)=\operatorname{clip}\bigl(\hat p_{\mathrm{LR}}(x)+g_{\mathrm{XGB-reg}}(x),\varepsilon,1-\varepsilon\bigr)}\]

**Important:** ordinary regression on the residuals optimizes squared residual error, not Bernoulli log loss. Clipping produces bounded probabilities but can introduce ties/poor calibration. Compare **AUC, average precision, log loss and Brier score** rather than assuming this is better. Original LR pipelines stay frozen. Each new XGBoost is fitted **only on its own outer-training fold**, and the held-out labels are never provided to that regressor.

In [ ]:
def actual_xgb_device(estimator):
    try:
        state=json.loads(estimator.get_booster().save_config())
        return state['learner']['generic_param']['device']
    except (KeyError,ValueError,AttributeError):
        return 'unverified'

def resolve_xgboost_device(request):
    if str(request).lower() in ('cpu','none'): return 'cpu'
    candidate = 'cuda:'+str(RESIDUAL_XGB_GPU_ID)
    demo_X=np.random.default_rng(RANDOM_STATE).normal(size=(180,4)).astype(np.float32)
    demo_y=np.random.default_rng(RANDOM_STATE+1).normal(size=180).astype(np.float32)
    try:
        with warnings.catch_warnings():
            warnings.simplefilter('ignore', UserWarning)
            probe=XGBRegressor(n_estimators=2,max_depth=2,tree_method='hist',
                               device=candidate,n_jobs=1).fit(demo_X,demo_y)
        real_device=actual_xgb_device(probe)
        if not str(real_device).startswith('cuda'):
            warnings.warn('XGBoost CUDA request trained on CPU; using CPU explicitly.')
            return 'cpu'
        return candidate
    except Exception as err:
        warnings.warn(f'CUDA probe failed ({type(err).__name__}); using CPU. {str(err)[:140]}')
        return 'cpu'

RESIDUAL_DEVICE = resolve_xgboost_device(RESIDUAL_XGB_DEVICE)
print('Effective residual-XGBoost device preference:',RESIDUAL_DEVICE)

# Stable configuration key prevents unintentional reuse if settings change.
source_info=(OUTPUT_DIR/'all_candidate_predictions.npz').stat()
source_signature={
    'source_size':source_info.st_size,
    'source_mtime_ns':source_info.st_mtime_ns,
    'baseline':BASELINE_LOGISTIC,
    'model_params':RESIDUAL_XGB_PARAMS,
    'stacking_params':RF_META_PARAMS,
    'stacking_feature_mode':STACKING_BASE_MODE,
    'n_train':len(y), 'n_test':len(test),
    'source_manifest':manifest,
}
SOURCE_KEY=hashlib.sha256(json.dumps(source_signature,sort_keys=True,default=str).encode()).hexdigest()[:16]
print('Current cached-run/configuration fingerprint:',SOURCE_KEY)

In [ ]:
def load_or_rebuild_logistic(fold):
    model_path=OUTPUT_DIR/'models'/f'{BASELINE_LOGISTIC}_fold{fold}.joblib'
    if not model_path.is_file():
        raise FileNotFoundError(
            f'Missing fitted logistic fold model: {model_path}. '
            'The first notebook saves it when SAVE_FOLD_MODELS=True. '
            'Do not rerun the full model sweep; recover this model from your existing outputs/models/ backup. '
            'If it was never saved, only this five-fold LR baseline must be rebuilt, separately.'
        )
    # Saved model is a built-in sklearn Pipeline for logit_C1; no custom notebook class required.
    return joblib.load(model_path)

def find_preprocess(model):
    if not hasattr(model,'named_steps') or 'preprocess' not in model.named_steps:
        raise TypeError('Expected a fitted logistic sklearn Pipeline with a preprocess step.')
    return model.named_steps['preprocess']

def save_stage_checkpoint(name, oof, per_fold_test):
    np.savez_compressed(EXT_DIR/f'{name}_checkpoint.npz',source_key=np.array(SOURCE_KEY),
                        oof=oof,per_fold_test=per_fold_test)

def load_stage_checkpoint(name):
    path=EXT_DIR/f'{name}_checkpoint.npz'
    if not path.exists() or RERUN_EXISTING_SECOND_STAGE: return None
    with np.load(path,allow_pickle=False) as z:
        if str(z['source_key'].item())!=SOURCE_KEY:
            warnings.warn(f'Ignoring stale checkpoint {name}: different original run or parameters.')
            return None
        a=np.asarray(z['oof'],dtype=np.float64)
        b=np.asarray(z['per_fold_test'],dtype=np.float64)
        if a.shape!=(len(y),) or b.shape!=(5,len(test)):
            raise AssertionError(f'Unexpected cached dimensions for {name}')
        return a,b

cached=load_stage_checkpoint('residual')
if cached is None:
    residual_oof=np.full(len(y),np.nan,dtype=np.float64)
    residual_test_by_fold=np.full((5,len(test)),np.nan,dtype=np.float64)
else:
    residual_oof,residual_test_by_fold=cached

for fold,(tr,va) in enumerate(fold_splits,1):
    if np.isfinite(residual_oof[va]).all() and np.isfinite(residual_test_by_fold[fold-1]).all():
        print(f'Residual fold {fold}: loaded from validated checkpoint.'); continue
    start=time.perf_counter()
    lr=load_or_rebuild_logistic(fold)
    # Strong row-order/fold-provenance check against the original saved OOF probabilities.
    sample_indices=va[:min(256,len(va))]
    sample_pred=lr.predict_proba(X.iloc[sample_indices])[:,1]
    if not np.allclose(sample_pred,predictions[BASELINE_LOGISTIC]['oof'][sample_indices],atol=1e-4,rtol=1e-4):
        raise AssertionError(f'Fold {fold} LR model predictions do not match cached OOF data. STOP: row order, folds or model mismatch.')
    lr_train=lr.predict_proba(X.iloc[tr])[:,1]
    residual_target=np.asarray(y[tr]-lr_train,dtype=np.float32)
    transformer=find_preprocess(lr)
    X_tr_t=np.asarray(transformer.transform(X.iloc[tr]),dtype=np.float32)
    X_va_t=np.asarray(transformer.transform(X.iloc[va]),dtype=np.float32)
    X_te_t=np.asarray(transformer.transform(X_test),dtype=np.float32)
    model_path=CACHE_DIR/f'explicit_residual_xgb_fold{fold}_{SOURCE_KEY}.joblib'
    if model_path.exists() and USE_SAVED_SECOND_STAGE_MODELS and not RERUN_EXISTING_SECOND_STAGE:
        reg=joblib.load(model_path)
    else:
        reg=XGBRegressor(**RESIDUAL_XGB_PARAMS,tree_method='hist',
                         device=RESIDUAL_DEVICE,n_jobs=N_JOBS,random_state=RANDOM_STATE+fold)
        try:
            reg.fit(X_tr_t,residual_target)
        except Exception as err:
            if not RESIDUAL_DEVICE.startswith('cuda'): raise
            warnings.warn(f'Residual XGBoost CUDA training failed in fold {fold}; retrying on CPU. {err}')
            reg.set_params(device='cpu'); reg.fit(X_tr_t,residual_target)
        joblib.dump(reg,model_path,compress=3)
    p_va=lr.predict_proba(X.iloc[va])[:,1]
    p_te=lr.predict_proba(X_test)[:,1]
    correction_valid=reg.predict(X_va_t)
    correction_test=reg.predict(X_te_t)
    residual_oof[va]=np.clip(p_va+correction_valid,1e-7,1-1e-7)
    residual_test_by_fold[fold-1]=np.clip(p_te+correction_test,1e-7,1-1e-7)
    save_stage_checkpoint('residual',residual_oof,residual_test_by_fold)
    print(f'Residual fold {fold}/5: AUC={roc_auc_score(y[va],residual_oof[va]):.5f}, '
          f'device={actual_xgb_device(reg)}, time={time.perf_counter()-start:.1f}s')
    del lr,reg,X_tr_t,X_va_t,X_te_t
assert np.isfinite(residual_oof).all() and np.isfinite(residual_test_by_fold).all()
results['ensemble3_explicit_residual']={'oof':residual_oof,'test':residual_test_by_fold.mean(axis=0)}
print('Explicit residual correction complete. Did not refit a previously saved classifier.')

## 5. Ensemble 4 — stacking with **only a random forest** as meta-model

Construct a level-1 design matrix using cached OOF probabilities:

\[Z_{im}=\hat p_m^{\mathrm{OOF}}(x_i),\qquad
\hat p_{\mathrm{stack}}(x)=RF_{\mathrm{meta}}\left(\hat p_1(x),\dots,\hat p_M(x)\right).\]

`Z` has **one column per standalone candidate** and contains **no original transaction predictors**: RF is the *only* meta-learner. The meta-forest trains on cached OOF predictions, not in-sample base fitted predictions. For a descriptive meta-CV curve, fit the meta-forest on four original folds of `Z`, predict the fifth, and repeat. For the final test prediction, refit a meta-forest on all OOF columns and apply it to the mean test probabilities already saved for the same candidates.

**Nontrivial limitation:** these meta-CV predictions are not equivalent to a genuinely nested CV stacking evaluation. The original base learner used some meta-validation records in training models that produced *other* meta-training rows. In addition, base test inputs are fold-averaged probabilities whereas training inputs are single-fold OOF probabilities. Treat the stacked OOF AUC as exploratory; a clean validation requires additional base-model training or a never-used holdout. Scikit-learn likewise recommends OOF training for the final stack estimator rather than fitting it to in-sample predictions.

In [ ]:
if STACKING_BASE_MODE=='best_per_family':
    stack_names=list(family_members)
elif STACKING_BASE_MODE=='all_standalone':
    stack_names=standalone['candidate'].astype(str).tolist()
else:
    raise ValueError('STACKING_BASE_MODE must be all_standalone or best_per_family.')
assert len(stack_names)>1
Z=np.column_stack([predictions[name]['oof'] for name in stack_names]).astype(np.float32)
Z_test=np.column_stack([predictions[name]['test'] for name in stack_names]).astype(np.float32)
assert Z.shape==(len(y),len(stack_names)) and Z_test.shape==(len(test),len(stack_names))
assert np.isfinite(Z).all() and np.isfinite(Z_test).all()
print('RF meta-inputs:',len(stack_names),'columns; OOF matrix:',Z.shape)
print('Only predictions (no original predictors):',stack_names)

cached=load_stage_checkpoint('rf_stack')
if cached is None:
    stack_oof=np.full(len(y),np.nan,dtype=np.float64)
    stack_test_by_fold=np.full((5,len(test)),np.nan,dtype=np.float64)
else:
    stack_oof,stack_test_by_fold=cached
rf_importance=[]
for fold,(tr,va) in enumerate(fold_splits,1):
    model_path=CACHE_DIR/f'rf_stack_fold{fold}_{SOURCE_KEY}.joblib'
    if model_path.exists() and USE_SAVED_SECOND_STAGE_MODELS and not RERUN_EXISTING_SECOND_STAGE:
        meta=joblib.load(model_path)
    elif (np.isfinite(stack_oof[va]).all()
          and np.isfinite(stack_test_by_fold[fold-1]).all()):
        print(f'Stacking fold {fold}: using saved predictions.'); continue
    else:
        meta=RandomForestClassifier(**RF_META_PARAMS,random_state=RANDOM_STATE+fold)
        meta.fit(Z[tr],y[tr])
        joblib.dump(meta,model_path,compress=3)
    stack_oof[va]=meta.predict_proba(Z[va])[:,1]
    # Diagnostic; the *final* test uses a meta-model trained on all OOF rows.
    stack_test_by_fold[fold-1]=meta.predict_proba(Z_test)[:,1]
    rf_importance.append(pd.DataFrame({'candidate':stack_names,'importance':meta.feature_importances_,'fold':fold}))
    save_stage_checkpoint('rf_stack',stack_oof,stack_test_by_fold)
    print(f'RF stacking fold {fold}/5: AUC={roc_auc_score(y[va],stack_oof[va]):.5f}')
    del meta
assert np.isfinite(stack_oof).all()
final_rf_path=CACHE_DIR/f'rf_stack_full_{SOURCE_KEY}.joblib'
if final_rf_path.exists() and USE_SAVED_SECOND_STAGE_MODELS and not RERUN_EXISTING_SECOND_STAGE:
    final_rf=joblib.load(final_rf_path)
else:
    final_rf=RandomForestClassifier(**RF_META_PARAMS,random_state=RANDOM_STATE)
    final_rf.fit(Z,y)
    joblib.dump(final_rf,final_rf_path,compress=3)
stack_test_final=final_rf.predict_proba(Z_test)[:,1]
results['ensemble4_rf_stacking']={'oof':stack_oof,'test':stack_test_final}
rf_importance.append(pd.DataFrame({'candidate':stack_names,'importance':final_rf.feature_importances_,'fold':'final'}))
importance_df=pd.concat(rf_importance,ignore_index=True)
importance_df.to_csv(EXT_DIR/'rf_meta_feature_importance.csv',index=False)
print('RF stacking complete; final test prediction uses all-OOF-trained meta model.')

## 6. Final five-way comparison and model selection

The report compares **exactly five** strategies: the best standalone baseline plus four ensemble architectures. Primary criterion is pooled OOF ROC AUC, exactly matching the Kaggle leaderboard metric. Supporting metrics are average precision, log loss, Brier score and top-1%-review recall/precision.

**Selection caution:** comparing multiple models on the same OOF predictions selects an optimistically biased maximum; score differences smaller than uncertainty or fold variation should not be over-interpreted. The residual model is outer-fold validated relative to its frozen LR; stacking has the additional cross-fitting limitation explained above. AUC is invariant to monotone score transforms, whereas log loss/Brier reflect probability quality. No claim of Kaggle leaderboard performance is made here.

In [ ]:
def score_probs(y_true,p):
    p=np.clip(np.asarray(p,dtype=np.float64),1e-7,1-1e-7)
    k=max(1,int(np.ceil(len(y_true)*0.01)))
    top=np.argpartition(p,-k)[-k:]
    return dict(roc_auc=roc_auc_score(y_true,p),
                average_precision=average_precision_score(y_true,p),
                log_loss=log_loss(y_true,p,labels=[0,1]),
                brier=brier_score_loss(y_true,p),
                precision_top1=float(y_true[top].mean()),
                recall_top1=float(y_true[top].sum()/max(y_true.sum(),1)))

labels={
    'best_single':'Best standalone',
    'ensemble1_blend':'1 — Probability blend',
    'ensemble2_logit_offset':'2 — Logistic-offset XGBoost',
    'ensemble3_explicit_residual':'3 — Explicit residual XGBoost',
    'ensemble4_rf_stacking':'4 — Random-forest stacking',
}
assert set(results)==set(labels)
comp=pd.DataFrame([{'strategy':name,'description':labels[name],**score_probs(y,record['oof'])}
                   for name,record in results.items()]).sort_values('roc_auc',ascending=False).reset_index(drop=True)
comp['auc_gain_vs_single']=comp['roc_auc']-float(comp.loc[comp['strategy']=='best_single','roc_auc'].iloc[0])
comp.to_csv(EXT_DIR/'five_way_comparison.csv',index=False)
display(comp.round(6))

winner=str(comp.iloc[0]['strategy'])
print('EXPLORATORY OOF winner:', labels[winner], '— ROC AUC:',round(comp.iloc[0]['roc_auc'],6))
print('No independent holdout has validated this winner. Choose final deployment only after external validation.')

### 6.1 Paired AUC differences with an illustrative stratified resampling interval

We compare each ensemble with the **same** best standalone model on paired records. For computational convenience, the bootstrap samples up to 2,000 fraud and 30,000 legitimate observations **with replacement** from the full OOF vectors, `BOOTSTRAP_REPS` times. It approximates **conditional variability**, not a fully valid uncertainty interval after model selection, nor does it account for dependence between transactions. Small differences are not reliable proof of superiority.

In [ ]:
if BOOTSTRAP_REPS>0:
    rng=np.random.default_rng(RANDOM_STATE+800)
    pos=np.flatnonzero(y==1); neg=np.flatnonzero(y==0)
    npick_pos=min(len(pos),2000); npick_neg=min(len(neg),30000)
    ref=results['best_single']['oof']
    records=[]
    for b in range(BOOTSTRAP_REPS):
        idx=np.r_[rng.choice(pos,npick_pos,replace=True), rng.choice(neg,npick_neg,replace=True)]
        base=roc_auc_score(y[idx],ref[idx])
        for name in labels:
            if name=='best_single': continue
            delta=roc_auc_score(y[idx],results[name]['oof'][idx])-base
            records.append({'replicate':b,'strategy':name,'auc_difference':delta})
    bootstrap=pd.DataFrame(records)
    intervals=(bootstrap.groupby('strategy')['auc_difference'].agg(
        lo=lambda v:np.quantile(v,.025),median='median',hi=lambda v:np.quantile(v,.975))
        .reset_index())
    intervals.to_csv(EXT_DIR/'paired_auc_difference_exploratory.csv',index=False)
    display(intervals.round(6))

## 7. Seaborn diagnostics: accuracy, ranking, reliability, and meta-model behavior

Every figure is labeled and saved in `outputs/ensemble_extension/figures/`. These figures are separate from the original 44 plots and will not replace them.

In [ ]:
fig,axs=plt.subplots(2,2,figsize=(16,11))
plot_order=comp.sort_values('roc_auc')
sns.barplot(data=plot_order,x='roc_auc',y='description',ax=axs[0,0],color='#0f766e')
axs[0,0].set(title='OOF ROC AUC — five-way comparison',xlabel='ROC AUC (higher is better)',ylabel='Strategy')
sns.barplot(data=comp.sort_values('average_precision'),x='average_precision',y='description',ax=axs[0,1],color='#346a9c')
axs[0,1].set(title='OOF average precision',xlabel='AP (higher is better)',ylabel='Strategy')
sns.barplot(data=comp.sort_values('log_loss',ascending=False),x='log_loss',y='description',ax=axs[1,0],color='#d17a46')
axs[1,0].set(title='OOF log loss',xlabel='Log loss (lower is better)',ylabel='Strategy')
sns.barplot(data=comp.sort_values('brier',ascending=False),x='brier',y='description',ax=axs[1,1],color='#a770a2')
axs[1,1].set(title='OOF Brier score',xlabel='Brier (lower is better)',ylabel='Strategy')
fig.suptitle('Cached standalone vs four ensemble architectures',y=1.01,fontsize=15)
fig.tight_layout()
fig.savefig(FIG_DIR/'01_five_way_metric_scorecard.png',bbox_inches='tight')
plt.show();plt.close(fig)

fig,axs=plt.subplots(1,2,figsize=(16,6))
for name in labels:
    fpr,tpr,_=roc_curve(y,results[name]['oof'])
    prec,rec,_=precision_recall_curve(y,results[name]['oof'])
    label=f'{labels[name]} (AUC {score_probs(y,results[name]["oof"])["roc_auc"]:.4f})'
    axs[0].plot(fpr,tpr,label=label,linewidth=1.7)
    axs[1].plot(rec,prec,label=f'{labels[name]} (AP {score_probs(y,results[name]["oof"])["average_precision"]:.4f})',linewidth=1.7)
axs[0].plot([0,1],[0,1],ls='--',color='gray',alpha=.4)
axs[1].axhline(y.mean(),ls='--',color='gray',label=f'Fraud prevalence {y.mean():.2%}')
for ax,title,xlab,ylab in [(axs[0],'Out-of-fold ROC curves','False positive rate','True positive rate'),
                          (axs[1],'Out-of-fold precision–recall curves','Recall','Precision')]:
    ax.set(title=title,xlabel=xlab,ylabel=ylab)
    ax.legend(fontsize=8,loc='best')
fig.tight_layout();fig.savefig(FIG_DIR/'02_roc_precision_recall.png',bbox_inches='tight')
plt.show();plt.close(fig)

In [ ]:
# Residual correction, paired change in probability, and stacking influence.
base=results['best_single']['oof']
logit=predictions[BASELINE_LOGISTIC]['oof']
resid=results['ensemble3_explicit_residual']['oof']
fig,axs=plt.subplots(1,3,figsize=(18,5))
rng=np.random.default_rng(RANDOM_STATE+9)
sample=rng.choice(len(y),min(12000,len(y)),replace=False)
axs[0].scatter(logit[sample],resid[sample],c=y[sample],alpha=.24,s=8,cmap='coolwarm',rasterized=True)
axs[0].plot([0,1],[0,1],ls='--',color='black',lw=1)
axs[0].set(xlabel='Baseline logistic fraud probability',ylabel='Explicit residual corrected probability',
           title='Probability changes from residual correction')
change=resid-logit
sns.histplot(change[sample],bins=65,ax=axs[1],color='#0f766e')
axs[1].axvline(0,color='black',linestyle='--',linewidth=1)
axs[1].set(xlabel='Corrected minus original probability',ylabel='Sampled transactions',
           title='Distribution of explicit residual corrections')
imp=importance_df[importance_df['fold'].eq('final')].nlargest(12,'importance').sort_values('importance')
sns.barplot(data=imp,x='importance',y='candidate',ax=axs[2],color='#346a9c')
axs[2].set(xlabel='RF impurity importance (descriptive)',ylabel='Base prediction',
           title='RF stacking: importance of base-model predictions')
fig.suptitle('Explain the two newly fitted ensemble approaches',fontsize=15,y=1.02)
fig.tight_layout();fig.savefig(FIG_DIR/'03_residual_corrections_and_stacking.png',bbox_inches='tight')
plt.show();plt.close(fig)

fold_rows=[]
for f,(_,va) in enumerate(fold_splits,1):
    for name in labels:
        fold_rows.append({'fold':f,'strategy':labels[name],
                          'roc_auc':roc_auc_score(y[va],results[name]['oof'][va])})
fold_table=pd.DataFrame(fold_rows)
fold_table.to_csv(EXT_DIR/'five_way_fold_metrics.csv',index=False)
fig,ax=plt.subplots(figsize=(13,5.5))
sns.pointplot(data=fold_table,x='fold',y='roc_auc',hue='strategy',
              markers='o',dodge=.30,errorbar=None,ax=ax)
ax.set(title='OOF ROC AUC by original stratified fold',xlabel='Held-out fold',
       ylabel='ROC AUC',ylim=(max(0,fold_table['roc_auc'].min()-.005),
                             min(1,fold_table['roc_auc'].max()+.005)))
ax.legend(fontsize=8,loc='best')
fig.tight_layout();fig.savefig(FIG_DIR/'04_fold_stability.png',bbox_inches='tight')
plt.show();plt.close(fig)

## 8. Export all five Kaggle submissions and preserve provenance

The selected primary submission uses **maximum apparent pooled OOF AUC**, not an independent test-set comparison. Every strategy gets its own submission CSV so you can compare the public leaderboard without losing your model choices. The `id,is_fraud` order follows Kaggle's `sample_submission.csv`. None of these files is submitted automatically.

The exact saved-model paths, column order for stacking, hyperparameters, project assumptions, versions and run fingerprint are saved in `selection_manifest.json`. **Do not upload the per-row prediction NPZ, model binaries, or competition CSVs to a public GitHub repository.**

In [ ]:
submission_names={
    'best_single':'submission_best_standalone.csv',
    'ensemble1_blend':'submission_blend.csv',
    'ensemble2_logit_offset':'submission_logit_offset.csv',
    'ensemble3_explicit_residual':'submission_explicit_residual.csv',
    'ensemble4_rf_stacking':'submission_rf_stacking.csv',
}
for strategy,filename in submission_names.items():
    prob=np.asarray(results[strategy]['test'],dtype=np.float64)
    assert prob.shape==(len(test),) and np.isfinite(prob).all()
    assert 0<=prob.min() and prob.max()<=1
    out=sample_submission[[ID]].copy()
    out[TARGET]=prob
    assert out[ID].equals(sample_submission[ID])
    out.to_csv(EXT_DIR/filename,index=False)
submission=sample_submission[[ID]].copy()
submission[TARGET]=results[winner]['test']
submission.to_csv(EXT_DIR/'submission_ensemble_selected.csv',index=False)
np.savez_compressed(EXT_DIR/'five_way_oof_and_test_predictions.npz',
                    **{f'{name}__{part}':p[part] for name,p in results.items() for part in ('oof','test')})
manifest_out={
    'task':'four-ensemble extension to LDS1071 Kaggle fraud notebook',
    'source_dir':str(OUTPUT_DIR.resolve()),
    'source_npz_file':'all_candidate_predictions.npz',
    'source_run_manifest':manifest,
    'fingerprint':SOURCE_KEY,
    'base_training_rows':len(y),'base_test_rows':len(test),
    'best_single':best_single,'blend_members':family_members,
    'cached_offset_model':offset_name,'explicit_residual_lr':BASELINE_LOGISTIC,
    'residual_regressor_params':RESIDUAL_XGB_PARAMS,'residual_device_requested':RESIDUAL_XGB_DEVICE,
    'residual_device_resolved':RESIDUAL_DEVICE,
    'rf_stacking_base_candidates':stack_names,'rf_meta_params':RF_META_PARAMS,
    'chosen_strategy_exploratory':winner,
    'primary_metric':'pooled OOF roc_auc',
    'validation_note':'OOF-based selection is optimistic; meta-CV of saved OOF features is not fully nested; independent holdout or full nested CV needed.',
    'software':{'scikit-learn':sklearn_version,'xgboost':xgboost.__version__,
                'numpy':np.__version__,'pandas':pd.__version__},
}
(EXT_DIR/'selection_manifest.json').write_text(json.dumps(manifest_out,indent=2,default=str),encoding='utf-8')
print('Primary exploratory submission:', EXT_DIR/'submission_ensemble_selected.csv')
print('Winner (conditional on cached OOF):',winner)
print('Other files saved in:',EXT_DIR.resolve())

## 9. Discussion and next steps

**What has actually been tested:** the original notebook has trained a collection of standalone and offset candidates. This notebook reads those existing predictions; averages a predefined set of them; trains a probability-residual XGBRegressor using cached fold-specific logistic models; trains a **random forest, and only a random forest**, to combine candidate OOF probability columns; creates a five-way metric table; then saves per-strategy submissions.

**Do not conflate these methods:**

- **Blending:** fixed average of different models' predicted probabilities. No learned meta-model.
- **Offset boosting:** the baseline logistic *logits* enter XGBoost via `base_margin`, and the binary logistic loss is optimized through gradients/Hessians; new trees repeatedly correct remaining logit-scale errors.
- **Explicit residual modeling:** XGBRegressor directly targets \(y-p_{LR}\) via squared error and adds predicted corrections to **probabilities**, then clips; no `base_margin` is passed.
- **Stacking:** a learned random forest receives **only the out-of-fold predicted probabilities from other models**, then predicts the fraud outcome. Not a second stage trained on the raw transaction columns.

**Limits:** some base-model candidates and blend members were selected by the same OOF scores used to rank ensembles; the same data was explored extensively before model selection. The RF meta-CV is an approximation based on reused OOF features, not fully nested external validation. Outlier prevalence and synthetically generated data may not generalize to live financial transactions. Compare multiple independent Kaggle/private scores conservatively and do not repeatedly tune to a public leaderboard. Commit only non-sensitive aggregate summaries once Kaggle competition sharing rules permit publication.

### Reproducibility

1. Keep the original `final_project/outputs/` intact; the source `run_manifest.json` must match your training data.
2. Use `conda activate lds1071-fraud`, `jupyter lab`, and the **Python (LDS1071 Fraud)** kernel.
3. Open `final_project/ensemble_comparison.ipynb` and select **Run All**.
4. Inspect `outputs/ensemble_extension/five_way_comparison.csv`, four ensemble submission CSVs, and figures.
5. If the baseline fold models are missing, recover them from your workstation's original `outputs/models/` backup. The second notebook refuses to silently rerun all original candidates.